# LOO Benchmark: CCA vs scANVI
Leave-one-out evaluation across all 11 Yost 2019 BCC patients.
Each fold: one patient held out as query, remaining 10 as reference.


In [ ]:
# Parameters — override with papermill -p
loo_dir          = "data/loo"
exhausted_labels = ["CD8_ex", "CD8_ex_act"]
pos_class        = "CD8_ex"
out_dir          = "benchmarking"


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
from IPython.display import display as ipy_display, Image
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score,
                             confusion_matrix, classification_report)
os.makedirs(out_dir, exist_ok=True)


## 1. Load LOO Fold CSVs

In [ ]:
cca_files  = sorted(glob.glob(os.path.join(loo_dir, "fold_*_cca.csv")))
scvi_files = sorted(glob.glob(os.path.join(loo_dir, "fold_*_scvi.csv")))
print(f"CCA folds:  {len(cca_files)}")
print(f"scVI folds: {len(scvi_files)}")

def load_folds(files, method):
    dfs = []
    for f in files:
        df = pd.read_csv(f, index_col=0)
        df.columns = df.columns.str.strip()
        df["method"] = method
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

cca_df  = load_folds(cca_files,  "CCA")
scvi_df = load_folds(scvi_files, "scANVI")
loo_df  = pd.concat([cca_df, scvi_df], ignore_index=True)

# Exhausted binary flag
loo_df["true_ex"] = loo_df["true_label"].isin(exhausted_labels)
loo_df["pred_ex"] = loo_df["pred_label"].isin(exhausted_labels)

print(f"\nTotal cells: {len(loo_df)}")
print(loo_df.groupby("method").size())


## 2. Per-patient Metrics

In [ ]:
patients = sorted(loo_df["patient"].unique())
metrics_rows = []
for method in ["CCA", "scANVI"]:
    for patient in patients:
        sub = loo_df[(loo_df["method"]==method) & (loo_df["patient"]==patient)]
        if len(sub) == 0:
            continue
        acc    = accuracy_score(sub["true_label"], sub["pred_label"])
        mf1    = f1_score(sub["true_label"], sub["pred_label"], average="macro", zero_division=0)
        wf1    = f1_score(sub["true_label"], sub["pred_label"], average="weighted", zero_division=0)
        cd8_p  = precision_score(sub["true_label"], sub["pred_label"],
                                 labels=[pos_class], average="micro", zero_division=0)
        cd8_r  = recall_score(sub["true_label"], sub["pred_label"],
                              labels=[pos_class], average="micro", zero_division=0)
        cd8_f1 = f1_score(sub["true_label"], sub["pred_label"],
                          labels=[pos_class], average="micro", zero_division=0)
        ex_f1  = f1_score(sub["true_ex"], sub["pred_ex"], zero_division=0)
        metrics_rows.append(dict(method=method, patient=patient,
                                 accuracy=acc, macro_f1=mf1, weighted_f1=wf1,
                                 cd8ex_prec=cd8_p, cd8ex_rec=cd8_r, cd8ex_f1=cd8_f1,
                                 exhausted_f1=ex_f1))

metrics_df = pd.DataFrame(metrics_rows)
out_csv = os.path.join(out_dir, "loo_per_patient_metrics.csv")
metrics_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
print(metrics_df.groupby("method")[["accuracy","macro_f1","cd8ex_f1"]].mean().round(3))


## 3. Grouped Bar Charts

In [ ]:
METRICS = ["accuracy","macro_f1","cd8ex_prec","cd8ex_rec","cd8ex_f1"]
COLORS  = {"CCA": "#4878cf", "scANVI": "#e05c4b"}

fig, axes = plt.subplots(len(METRICS), 1, figsize=(13, 2.8*len(METRICS)))
x = np.arange(len(patients))
w = 0.35

for ax, metric in zip(axes, METRICS):
    for i, method in enumerate(["CCA","scANVI"]):
        vals = [metrics_df[(metrics_df.method==method) &
                           (metrics_df.patient==pt)][metric].values[0]
                if len(metrics_df[(metrics_df.method==method) &
                                  (metrics_df.patient==pt)]) > 0
                else np.nan
                for pt in patients]
        offset = (i - 0.5) * w
        ax.bar(x + offset, vals, w, label=method, color=COLORS[method], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(patients, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel(metric, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.set_title(f"{metric} per patient", fontsize=10)

plt.tight_layout()
out_png = os.path.join(out_dir, "loo_bar_charts.png")
plt.savefig(out_png, dpi=120, bbox_inches="tight")
buf = BytesIO()
plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
buf.seek(0)
ipy_display(Image(data=buf.read()))
plt.close()
print(f"Saved: {out_png}")


## 4. CCA vs scANVI Scatter (per-patient)

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(4*len(METRICS), 4))
for ax, metric in zip(axes, METRICS):
    cca_vals  = [metrics_df[(metrics_df.method=="CCA")    & (metrics_df.patient==pt)][metric].values[0]
                 if len(metrics_df[(metrics_df.method=="CCA")    & (metrics_df.patient==pt)]) > 0 else np.nan
                 for pt in patients]
    scvi_vals = [metrics_df[(metrics_df.method=="scANVI") & (metrics_df.patient==pt)][metric].values[0]
                 if len(metrics_df[(metrics_df.method=="scANVI") & (metrics_df.patient==pt)]) > 0 else np.nan
                 for pt in patients]
    ax.scatter(cca_vals, scvi_vals, color="#555", zorder=5)
    for pt, cx, sx in zip(patients, cca_vals, scvi_vals):
        ax.annotate(pt, (cx, sx), fontsize=7, ha="center", va="bottom")
    lim = [min(cca_vals+scvi_vals)-0.05, max(cca_vals+scvi_vals)+0.05]
    ax.plot(lim, lim, "k--", lw=0.8, alpha=0.5)
    ax.set_xlabel("CCA", fontsize=9)
    ax.set_ylabel("scANVI", fontsize=9)
    ax.set_title(metric, fontsize=9)
    ax.set_xlim(lim); ax.set_ylim(lim)

plt.suptitle("CCA vs scANVI per patient", fontsize=11, y=1.02)
plt.tight_layout()
out_png = os.path.join(out_dir, "loo_scatter.png")
plt.savefig(out_png, dpi=120, bbox_inches="tight")
buf = BytesIO()
plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
buf.seek(0)
ipy_display(Image(data=buf.read()))
plt.close()
print(f"Saved: {out_png}")


## 5. Box Plots & Aggregated Confusion Matrices

In [ ]:
# Box plots
fig, axes = plt.subplots(1, len(METRICS), figsize=(3.5*len(METRICS), 4))
for ax, metric in zip(axes, METRICS):
    data_cca  = metrics_df[metrics_df.method=="CCA"][metric].values
    data_scvi = metrics_df[metrics_df.method=="scANVI"][metric].values
    bp = ax.boxplot([data_cca, data_scvi], labels=["CCA","scANVI"],
                    patch_artist=True, widths=0.5,
                    medianprops=dict(color="black", lw=2))
    for patch, color in zip(bp["boxes"], [COLORS["CCA"], COLORS["scANVI"]]):
        patch.set_facecolor(color); patch.set_alpha(0.75)
    ax.set_title(metric, fontsize=9)
    ax.set_ylim(0, 1.05)

plt.suptitle("Distribution across patients", fontsize=11, y=1.02)
plt.tight_layout()
out_png = os.path.join(out_dir, "loo_boxplots.png")
plt.savefig(out_png, dpi=120, bbox_inches="tight")
buf = BytesIO()
plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
buf.seek(0)
ipy_display(Image(data=buf.read()))
plt.close()
print(f"Saved: {out_png}")


In [ ]:
# Aggregated confusion matrices
all_labels = sorted(loo_df["true_label"].unique())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, method in zip(axes, ["CCA","scANVI"]):
    sub = loo_df[loo_df.method==method]
    cm  = confusion_matrix(sub["true_label"], sub["pred_label"], labels=all_labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, xticklabels=all_labels, yticklabels=all_labels,
                annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                ax=ax, annot_kws={"size": 7})
    ax.set_title(f"{method} — row-normalised", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

plt.tight_layout()
out_png = os.path.join(out_dir, "loo_confusion_matrices.png")
plt.savefig(out_png, dpi=120, bbox_inches="tight")
buf = BytesIO()
plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
buf.seek(0)
ipy_display(Image(data=buf.read()))
plt.close()
print(f"Saved: {out_png}")


## 6. Summary Table

In [ ]:
summary = metrics_df.groupby("method")[METRICS].agg(["mean","std"]).round(4)
print(summary.to_string())
out_csv = os.path.join(out_dir, "loo_summary.csv")
summary.to_csv(out_csv)
print(f"\nSaved: {out_csv}")
